**Other autoreload modes, for reference:**

| Mode | New name   | Behavior                                                    |
|------|------------|-------------------------------------------------------------|
| `0`  | `off`      | Disabled                                                    |
| `1`  | `explicit` | Reloads only modules you mark with `%aimport`               |
| `2`  | `all`      | Reloads everything, which is the common choice              |
| `3`  | `complete` | Same as 2/all, but also adds any new objects in the module  |

In [ ]:
# Loads IPython's built-in autoreload extension. The extension reloads modules
# automatically before entering the execution of code typed at the IPython prompt.
%load_ext autoreload

# sets the mode. In mode 2, the extension will reload all modules (except those
# excluded by %aimport) every time before executing the Python code typed.
%autoreload 2

# Example. Say your notebook runs from pipeline.transform import clean_1099 and you
# fix a bug in transform.py in VS Code. Without autoreload, the kernel keeps using the
# old code until you restart it and re-run every cell. With autoreload, the next cell
# you run uses the fixed version.

import duckdb

from learning_journey.paths import DATA_DIR, SQL_DIR, project_root

data_file = DATA_DIR / "market_seed" / "bars_daily.csv"

In [ ]:
# Query file: 05_bar_windows.sql

# run it with uv run python -c "import duckdb, pathlib;
# print(duckdb.sql(pathlib.Path('sql/twins/day32_bar_windows.sql').read_text()))"

query_file = SQL_DIR / "05_bar_windows.sql"

# duckdb.connect() with no file name is an in-memory database.
conn = duckdb.connect()

# This works from any folder, in notebooks, in CI and in Docker, and the same .sql
# file can point at sample data or real data without being edited. That's the
# “configuration outside the code” habit.
conn.execute("SET VARIABLE bars_csv = $p", {"p": str(data_file)})
print(conn.sql(query_file.read_text()))
conn.close()
# project_root.cache_clear() clears cache and its statistics
print(f"project_root calls: {project_root.cache_info()}")

In [ ]:
# Query file: 06_gaps_and_islands_bars.sql

query_file = SQL_DIR / "06_gaps_and_islands_bars.sql"

conn = duckdb.connect()

conn.execute("SET VARIABLE bars_csv = $p", {"p": str(data_file)})
print(conn.sql(query_file.read_text()))
conn.close()

In [ ]:
conn = duckdb.connect()
conn.execute("SET VARIABLE bars_csv = $p", {"p": str(data_file)})

In [ ]:
query_1 = """
SELECT
        ticker,
        session_date,
        -- Make is_dry true only when the frame holds exactly 20 non-null volumes.
        ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY session_date) AS session_n,
        COUNT(volume) OVER w = 20
            AND volume < 0.5 * AVG(volume) OVER w AS is_dry
    FROM read_csv(getvariable('bars_csv'))
    WINDOW w AS (
        PARTITION BY ticker
        ORDER BY session_date
        ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
    );"""

print(conn.sql(query_1))

In [ ]:
query_2 = """
WITH flagged AS (
    SELECT
            ticker,
            session_date,
            ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY session_date) AS session_n,
            volume < 0.5 * AVG(volume) OVER (
                PARTITION BY ticker
                ORDER BY session_date
                ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
            ) AS is_dry
        FROM read_csv(getvariable('bars_csv'))
)
-- consecutive dry sessions share one value of (session_n - row number among dry rows)
SELECT
        ticker,
        session_date,
        session_n - ROW_NUMBER() OVER (
            PARTITION BY ticker
            ORDER BY session_date
        ) AS island
    FROM flagged
    WHERE is_dry;"""

print(conn.sql(query_2))

In [ ]:
query_3 = """
WITH flagged AS (   -- query_1
    SELECT
            ticker,
            session_date,
            ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY session_date) AS session_n,
            volume < 0.5 * AVG(volume) OVER (
                PARTITION BY ticker
                ORDER BY session_date
                ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
            ) AS is_dry
        FROM read_csv(getvariable('bars_csv'))
),
dry AS (            -- query_2
    -- consecutive dry sessions share one value of (session_n - row number among dry rows)
    SELECT
            ticker,
            session_date,
            session_n - ROW_NUMBER() OVER (
                PARTITION BY ticker
                ORDER BY session_date
            ) AS island
        FROM flagged
        WHERE is_dry
),
                   -- query_3
SELECT
        ticker,
        island,
        MIN(session_date) AS run_start,
        MAX(session_date) AS run_end,
        COUNT(*) AS dry_sessions
    FROM dry
    GROUP BY ticker, island;"""

print(conn.sql(query_3))

In [ ]:
conn.close()